In [ ]:
# EduFin Loan Data Engineering Project
# Technology: Python + PySpark
# Environment: Google Colab

print("EduFin Loan Data Engineering Project Started 🚀")

EduFin Loan Data Engineering Project Started 🚀


In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EduFinLoanProject") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.4


In [ ]:
data = [
    (1001, 25, "Bangalore", 500000, 25000, 24, "Salaried", 720, "Paid"),
    (1002, 32, "Mumbai", 800000, 40000, 36, "Salaried", 680, "Default"),
    (1003, 29, "Delhi", 450000, 22000, 24, "Self-Employed", 750, "Paid"),
    (1004, 41, "Bangalore", 1200000, 60000, 48, "Business", 640, "Default"),
    (1005, 35, "Chennai", 700000, 35000, 36, "Salaried", 710, "Paid"),
    (1006, 27, "Hyderabad", 300000, 15000, 18, "Self-Employed", 690, "Paid"),
    (1007, 45, "Mumbai", 1500000, 75000, 60, "Business", 610, "Default"),
    (1008, 31, "Delhi", 600000, 30000, 30, "Salaried", 730, "Paid"),
    (1009, 38, "Bangalore", 900000, 45000, 42, "Salaried", 670, "Default"),
    (1010, 26, "Chennai", 350000, 18000, 20, "Salaried", 760, "Paid"),
]

columns = [
    "loan_id",
    "age",
    "city",
    "loan_amount",
    "monthly_income",
    "loan_term_months",
    "employment_type",
    "credit_score",
    "loan_status"
]

df = spark.createDataFrame(data, columns)

df.show()

+-------+---+---------+-----------+--------------+----------------+---------------+------------+-----------+
|loan_id|age|     city|loan_amount|monthly_income|loan_term_months|employment_type|credit_score|loan_status|
+-------+---+---------+-----------+--------------+----------------+---------------+------------+-----------+
|   1001| 25|Bangalore|     500000|         25000|              24|       Salaried|         720|       Paid|
|   1002| 32|   Mumbai|     800000|         40000|              36|       Salaried|         680|    Default|
|   1003| 29|    Delhi|     450000|         22000|              24|  Self-Employed|         750|       Paid|
|   1004| 41|Bangalore|    1200000|         60000|              48|       Business|         640|    Default|
|   1005| 35|  Chennai|     700000|         35000|              36|       Salaried|         710|       Paid|
|   1006| 27|Hyderabad|     300000|         15000|              18|  Self-Employed|         690|       Paid|
|   1007| 45|   Mum

In [ ]:
print("Total records:", df.count())

print("Columns:", df.columns)

df.printSchema()

Total records: 10
Columns: ['loan_id', 'age', 'city', 'loan_amount', 'monthly_income', 'loan_term_months', 'employment_type', 'credit_score', 'loan_status']
root
 |-- loan_id: long (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- loan_amount: long (nullable = true)
 |-- monthly_income: long (nullable = true)
 |-- loan_term_months: long (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- credit_score: long (nullable = true)
 |-- loan_status: string (nullable = true)



In [ ]:
from pyspark.sql.functions import col, sum

missing_values = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_values.show()

+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+
|loan_id|age|city|loan_amount|monthly_income|loan_term_months|employment_type|credit_score|loan_status|
+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+
|      0|  0|   0|          0|             0|               0|              0|           0|          0|
+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+



In [ ]:
total_records = df.count()
unique_records = df.dropDuplicates().count()

print("Total records:", total_records)
print("Unique records:", unique_records)
print("Duplicate records:", total_records - unique_records)

Total records: 10
Unique records: 10
Duplicate records: 0


In [ ]:
from pyspark.sql.functions import count

duplicate_loan_ids = df.groupBy("loan_id") \
    .count() \
    .filter(col("count") > 1)

duplicate_loan_ids.show()

+-------+-----+
|loan_id|count|
+-------+-----+
+-------+-----+



In [ ]:
df.filter(col("loan_amount") <= 0).show()

+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+
|loan_id|age|city|loan_amount|monthly_income|loan_term_months|employment_type|credit_score|loan_status|
+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+
+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+



In [ ]:
df.filter(col("monthly_income") <= 0).show()

+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+
|loan_id|age|city|loan_amount|monthly_income|loan_term_months|employment_type|credit_score|loan_status|
+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+
+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+



In [ ]:
df.filter(
    (col("credit_score") < 300) |
    (col("credit_score") > 850)
).show()

+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+
|loan_id|age|city|loan_amount|monthly_income|loan_term_months|employment_type|credit_score|loan_status|
+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+
+-------+---+----+-----------+--------------+----------------+---------------+------------+-----------+



In [ ]:
from pyspark.sql.functions import round

df_transformed = df.withColumn(
    "loan_exposure_ratio",
    round(
        col("loan_amount") /
        (col("monthly_income") * col("loan_term_months")),
        2
    )
)

df_transformed.select(
    "loan_id",
    "loan_amount",
    "monthly_income",
    "loan_term_months",
    "loan_exposure_ratio"
).show()

+-------+-----------+--------------+----------------+-------------------+
|loan_id|loan_amount|monthly_income|loan_term_months|loan_exposure_ratio|
+-------+-----------+--------------+----------------+-------------------+
|   1001|     500000|         25000|              24|               0.83|
|   1002|     800000|         40000|              36|               0.56|
|   1003|     450000|         22000|              24|               0.85|
|   1004|    1200000|         60000|              48|               0.42|
|   1005|     700000|         35000|              36|               0.56|
|   1006|     300000|         15000|              18|               1.11|
|   1007|    1500000|         75000|              60|               0.33|
|   1008|     600000|         30000|              30|               0.67|
|   1009|     900000|         45000|              42|               0.48|
|   1010|     350000|         18000|              20|               0.97|
+-------+-----------+--------------+--

In [ ]:
df_transformed = df_transformed.withColumn(
    "monthly_loan_burden",
    round(
        col("loan_amount") / col("loan_term_months"),
        2
    )
)

df_transformed.select(
    "loan_id",
    "loan_amount",
    "loan_term_months",
    "monthly_loan_burden"
).show()

+-------+-----------+----------------+-------------------+
|loan_id|loan_amount|loan_term_months|monthly_loan_burden|
+-------+-----------+----------------+-------------------+
|   1001|     500000|              24|           20833.33|
|   1002|     800000|              36|           22222.22|
|   1003|     450000|              24|            18750.0|
|   1004|    1200000|              48|            25000.0|
|   1005|     700000|              36|           19444.44|
|   1006|     300000|              18|           16666.67|
|   1007|    1500000|              60|            25000.0|
|   1008|     600000|              30|            20000.0|
|   1009|     900000|              42|           21428.57|
|   1010|     350000|              20|            17500.0|
+-------+-----------+----------------+-------------------+



In [ ]:
from pyspark.sql.functions import when

df_risk = df_transformed.withColumn(
    "risk_category",
    when(
        (col("credit_score") < 650) |
        (col("loan_exposure_ratio") > 1.0),
        "High Risk"
    ).when(
        (col("credit_score") < 700) |
        (col("loan_exposure_ratio") > 0.5),
        "Medium Risk"
    ).otherwise("Low Risk")
)

df_risk.select(
    "loan_id",
    "credit_score",
    "loan_exposure_ratio",
    "loan_status",
    "risk_category"
).show()

+-------+------------+-------------------+-----------+-------------+
|loan_id|credit_score|loan_exposure_ratio|loan_status|risk_category|
+-------+------------+-------------------+-----------+-------------+
|   1001|         720|               0.83|       Paid|  Medium Risk|
|   1002|         680|               0.56|    Default|  Medium Risk|
|   1003|         750|               0.85|       Paid|  Medium Risk|
|   1004|         640|               0.42|    Default|    High Risk|
|   1005|         710|               0.56|       Paid|  Medium Risk|
|   1006|         690|               1.11|       Paid|    High Risk|
|   1007|         610|               0.33|    Default|    High Risk|
|   1008|         730|               0.67|       Paid|  Medium Risk|
|   1009|         670|               0.48|    Default|  Medium Risk|
|   1010|         760|               0.97|       Paid|  Medium Risk|
+-------+------------+-------------------+-----------+-------------+



In [ ]:
risk_summary = df_risk.groupBy("risk_category") \
    .count() \
    .orderBy("risk_category")

risk_summary.show()

+-------------+-----+
|risk_category|count|
+-------------+-----+
|    High Risk|    3|
|  Medium Risk|    7|
+-------------+-----+



In [ ]:
city_summary = df_risk.groupBy("city").agg(
    count("*").alias("total_loans"),
    sum(when(col("loan_status") == "Default", 1).otherwise(0)).alias("default_loans")
)

city_summary.show()

+---------+-----------+-------------+
|     city|total_loans|default_loans|
+---------+-----------+-------------+
|Bangalore|          3|            2|
|  Chennai|          2|            0|
|   Mumbai|          2|            2|
|    Delhi|          2|            0|
|Hyderabad|          1|            0|
+---------+-----------+-------------+



In [ ]:
city_summary = city_summary.withColumn(
    "default_rate",
    round(
        (col("default_loans") / col("total_loans")) * 100,
        2
    )
)

city_summary.orderBy(
    col("default_rate").desc()
).show()

+---------+-----------+-------------+------------+
|     city|total_loans|default_loans|default_rate|
+---------+-----------+-------------+------------+
|   Mumbai|          2|            2|       100.0|
|Bangalore|          3|            2|       66.67|
|  Chennai|          2|            0|         0.0|
|    Delhi|          2|            0|         0.0|
|Hyderabad|          1|            0|         0.0|
+---------+-----------+-------------+------------+



In [ ]:
from pyspark.sql.functions import avg, max, min

portfolio_summary = df_risk.agg(
    count("*").alias("total_loans"),
    sum(when(col("loan_status") == "Default", 1).otherwise(0)).alias("default_loans"),
    sum(col("loan_amount")).alias("total_loan_amount"),
    avg(col("loan_amount")).alias("average_loan_amount")
)

portfolio_summary.show()

+-----------+-------------+-----------------+-------------------+
|total_loans|default_loans|total_loan_amount|average_loan_amount|
+-----------+-------------+-----------------+-------------------+
|         10|            4|          7300000|           730000.0|
+-----------+-------------+-----------------+-------------------+



In [ ]:
portfolio_summary = portfolio_summary.withColumn(
    "default_rate",
    round(
        (col("default_loans") / col("total_loans")) * 100,
        2
    )
)

portfolio_summary.show()

+-----------+-------------+-----------------+-------------------+------------+
|total_loans|default_loans|total_loan_amount|average_loan_amount|default_rate|
+-----------+-------------+-----------------+-------------------+------------+
|         10|            4|          7300000|           730000.0|        40.0|
+-----------+-------------+-----------------+-------------------+------------+



In [ ]:
employment_summary = df_risk.groupBy("employment_type").agg(
    count("*").alias("total_loans"),
    sum(when(col("loan_status") == "Default", 1).otherwise(0)).alias("default_loans"),
    sum(col("loan_amount")).alias("total_loan_amount")
)

employment_summary = employment_summary.withColumn(
    "default_rate",
    round(
        (col("default_loans") / col("total_loans")) * 100,
        2
    )
)

employment_summary.orderBy(
    col("default_rate").desc()
).show()

+---------------+-----------+-------------+-----------------+------------+
|employment_type|total_loans|default_loans|total_loan_amount|default_rate|
+---------------+-----------+-------------+-----------------+------------+
|       Business|          2|            2|          2700000|       100.0|
|       Salaried|          6|            2|          3850000|       33.33|
|  Self-Employed|          2|            0|           750000|         0.0|
+---------------+-----------+-------------+-----------------+------------+



In [ ]:
risk_analysis = df_risk.groupBy("risk_category").agg(
    count("*").alias("total_loans"),
    sum(when(col("loan_status") == "Default", 1).otherwise(0)).alias("default_loans"),
    sum(col("loan_amount")).alias("total_exposure")
)

risk_analysis = risk_analysis.withColumn(
    "default_rate",
    round(
        (col("default_loans") / col("total_loans")) * 100,
        2
    )
)

risk_analysis.orderBy(
    col("risk_category")
).show()

+-------------+-----------+-------------+--------------+------------+
|risk_category|total_loans|default_loans|total_exposure|default_rate|
+-------------+-----------+-------------+--------------+------------+
|    High Risk|          3|            2|       3000000|       66.67|
|  Medium Risk|          7|            2|       4300000|       28.57|
+-------------+-----------+-------------+--------------+------------+



In [ ]:
final_df = df_risk.select(
    "loan_id",
    "age",
    "city",
    "employment_type",
    "monthly_income",
    "loan_amount",
    "loan_term_months",
    "credit_score",
    "loan_status",
    "loan_exposure_ratio",
    "monthly_loan_burden",
    "risk_category"
)

final_df.show()

+-------+---+---------+---------------+--------------+-----------+----------------+------------+-----------+-------------------+-------------------+-------------+
|loan_id|age|     city|employment_type|monthly_income|loan_amount|loan_term_months|credit_score|loan_status|loan_exposure_ratio|monthly_loan_burden|risk_category|
+-------+---+---------+---------------+--------------+-----------+----------------+------------+-----------+-------------------+-------------------+-------------+
|   1001| 25|Bangalore|       Salaried|         25000|     500000|              24|         720|       Paid|               0.83|           20833.33|  Medium Risk|
|   1002| 32|   Mumbai|       Salaried|         40000|     800000|              36|         680|    Default|               0.56|           22222.22|  Medium Risk|
|   1003| 29|    Delhi|  Self-Employed|         22000|     450000|              24|         750|       Paid|               0.85|            18750.0|  Medium Risk|
|   1004| 41|Bangalore

In [ ]:
print("Final record count:", final_df.count())

Final record count: 10


In [ ]:
final_df.write \
    .mode("overwrite") \
    .parquet("edufin_loan_curated")

In [ ]:
import os

print(os.listdir("edufin_loan_curated"))

['.part-00000-af2d6e64-931f-49a5-9161-ddea3b57c277-c000.snappy.parquet.crc', '.part-00001-af2d6e64-931f-49a5-9161-ddea3b57c277-c000.snappy.parquet.crc', 'part-00001-af2d6e64-931f-49a5-9161-ddea3b57c277-c000.snappy.parquet', '._SUCCESS.crc', 'part-00000-af2d6e64-931f-49a5-9161-ddea3b57c277-c000.snappy.parquet', '_SUCCESS']


In [ ]:
final_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("edufin_loan_curated_csv")

In [ ]:
print(os.listdir("edufin_loan_curated_csv"))

['part-00000-fa698ea3-6895-42f7-8798-0e53a0ef4901-c000.csv', '.part-00000-fa698ea3-6895-42f7-8798-0e53a0ef4901-c000.csv.crc', '._SUCCESS.crc', '_SUCCESS']


In [ ]:
parquet_df = spark.read.parquet("edufin_loan_curated")

parquet_df.show()
print("Parquet records:", parquet_df.count())

+-------+---+---------+---------------+--------------+-----------+----------------+------------+-----------+-------------------+-------------------+-------------+
|loan_id|age|     city|employment_type|monthly_income|loan_amount|loan_term_months|credit_score|loan_status|loan_exposure_ratio|monthly_loan_burden|risk_category|
+-------+---+---------+---------------+--------------+-----------+----------------+------------+-----------+-------------------+-------------------+-------------+
|   1001| 25|Bangalore|       Salaried|         25000|     500000|              24|         720|       Paid|               0.83|           20833.33|  Medium Risk|
|   1002| 32|   Mumbai|       Salaried|         40000|     800000|              36|         680|    Default|               0.56|           22222.22|  Medium Risk|
|   1003| 29|    Delhi|  Self-Employed|         22000|     450000|              24|         750|       Paid|               0.85|            18750.0|  Medium Risk|
|   1004| 41|Bangalore

In [ ]:
parquet_df.printSchema()

root
 |-- loan_id: long (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- monthly_income: long (nullable = true)
 |-- loan_amount: long (nullable = true)
 |-- loan_term_months: long (nullable = true)
 |-- credit_score: long (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- loan_exposure_ratio: double (nullable = true)
 |-- monthly_loan_burden: double (nullable = true)
 |-- risk_category: string (nullable = true)



# EduFin Loan Data Engineering Project

## 📌 Project Overview

This project demonstrates an end-to-end data engineering pipeline using PySpark to process and analyze loan data.

The project focuses on data quality validation, transformation, risk classification, aggregation, and generation of curated datasets in CSV and Parquet formats.

> Note: This is an independently built EduFin-style project created for portfolio and learning purposes. It is not presented as the official SkillAI case study.

---

## 🛠️ Technologies Used

- Python
- PySpark
- Apache Spark
- Google Colab
- CSV
- Parquet
- GitHub

---

## 🔄 Data Engineering Pipeline

Raw Loan Data  
↓  
PySpark DataFrame  
↓  
Data Quality Validation  
↓  
Data Transformation  
↓  
Risk Classification  
↓  
Business Aggregation  
↓  
Curated Dataset  
↓  
CSV + Parquet

---

## 🔍 Data Quality Checks

The following validations were performed:

- Missing value validation
- Duplicate record validation
- Duplicate loan ID validation
- Loan amount validation
- Monthly income validation
- Credit score range validation

---

## ⚙️ Transformations

### Loan Exposure Ratio

Calculated using:

Loan Amount / (Monthly Income × Loan Term)

### Monthly Loan Burden

Calculated using:

Loan Amount / Loan Term

### Risk Classification

Loans were classified into:

- High Risk
- Medium Risk
- Low Risk

based on credit score and loan exposure ratio.

---

## 📊 Business Analysis

The project generated:

- Overall loan portfolio statistics
- Overall default rate
- City-level loan and default analysis
- Employment-type default analysis
- Risk-category analysis
- Total loan exposure

---

## 📦 Output

The final curated dataset was exported in:

- CSV format
- Parquet format

The Parquet output was also read back into PySpark to validate the generated dataset.

---

## 🎯 Key PySpark Concepts Practiced

- SparkSession
- DataFrames
- Schema inspection
- `select()`
- `filter()`
- `withColumn()`
- `when()`
- `groupBy()`
- `agg()`
- `count()`
- `sum()`
- `avg()`
- `round()`
- Sorting
- CSV writing
- Parquet writing
- Reading Parquet
- Data validation

---

## 💡 Project Outcome

This project demonstrates how raw loan data can be transformed into a validated and business-ready dataset using PySpark.

The pipeline combines data-quality checks, feature engineering, risk classification, aggregation, and distributed data processing concepts.